In [ ]:
https://github.com/thanhliem-top/TRI-TUE-NHAN-TAO.git

In [ ]:
import random

# =========================
# 1. Cấu hình bài toán
# =========================

START = (
    (2, 8, 3),
    (1, 6, 4),
    (7, 0, 5)
)

GOAL = (
    (1, 2, 3),
    (8, 0, 4),
    (7, 6, 5)
)

# Thứ tự ưu tiên sinh trạng thái
# L = Left, R = Right, U = Up, D = Down
MOVES = ["L", "R", "U", "D"]


# =========================
# 2. Hàm in state
# =========================

def print_state(state):
    for row in state:
        print(row)
    print()


# =========================
# 3. Tìm vị trí của số 0
# =========================

def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j


# =========================
# 4. Chuyển tuple thành list để đổi chỗ
# =========================

def swap_position(state, r1, c1, r2, c2):
    new_state = [list(row) for row in state]

    new_state[r1][c1], new_state[r2][c2] = new_state[r2][c2], new_state[r1][c1]

    return tuple(tuple(row) for row in new_state)


# =========================
# 5. Sinh trạng thái lân cận
# =========================

def get_neighbors(state):
    neighbors = []

    zero_r, zero_c = find_zero(state)

    for move in MOVES:
        if move == "L":
            new_r, new_c = zero_r, zero_c - 1
        elif move == "R":
            new_r, new_c = zero_r, zero_c + 1
        elif move == "U":
            new_r, new_c = zero_r - 1, zero_c
        elif move == "D":
            new_r, new_c = zero_r + 1, zero_c

        # Kiểm tra xem vị trí mới có nằm trong ma trận 3x3 không
        if 0 <= new_r < 3 and 0 <= new_c < 3:
            new_state = swap_position(state, zero_r, zero_c, new_r, new_c)
            neighbors.append((move, new_state))

    return neighbors


# =========================
# 6. Tạo bảng vị trí goal
# =========================

def build_goal_positions(goal):
    positions = {}

    for i in range(3):
        for j in range(3):
            value = goal[i][j]
            positions[value] = (i, j)

    return positions


GOAL_POSITIONS = build_goal_positions(GOAL)


# =========================
# 7. Hàm heuristic Manhattan
# =========================

def manhattan(state):
    total = 0

    for i in range(3):
        for j in range(3):
            value = state[i][j]

            # Không tính ô trống 0
            if value != 0:
                goal_i, goal_j = GOAL_POSITIONS[value]

                distance = abs(i - goal_i) + abs(j - goal_j)

                total += distance

    return total


# =========================
# 8. Local Beam Search
# =========================

def local_beam_search(start, goal, k=2, max_steps=50):
    """
    Local Beam Search:
    - Giữ k trạng thái hiện tại cùng lúc.
    - Sinh tất cả hàng xóm từ k trạng thái đó.
    - Chọn k trạng thái tốt nhất theo h(n).
    """

    # Với bài học trên lớp: Current_State_set có k trạng thái.
    # Ở đây ta bắt đầu bằng start, rồi sinh thêm một vài trạng thái lân cận
    # để đủ k trạng thái ban đầu.
    current_state_set = [start]

    # Sinh thêm trạng thái ban đầu nếu k > 1
    start_neighbors = get_neighbors(start)

    for move, neighbor in start_neighbors:
        if len(current_state_set) < k:
            current_state_set.append(neighbor)

    # Nếu vẫn chưa đủ k thì cứ giữ số lượng đang có
    print("===== KHỞI TẠO =====")
    for index, state in enumerate(current_state_set):
        print(f"State {index + 1}: h = {manhattan(state)}")
        print_state(state)

    # Lặp tối đa max_steps bước để tránh vòng lặp vô hạn
    for step in range(max_steps):
        print(f"===== BƯỚC {step + 1} =====")

        neighbor_states = []

        # Sinh tất cả hàng xóm từ mỗi state trong Current_State_set
        for state_index, state in enumerate(current_state_set):
            print(f"Sinh hàng xóm từ State {state_index + 1}, h = {manhattan(state)}")

            neighbors = get_neighbors(state)

            for move, neighbor in neighbors:
                h_value = manhattan(neighbor)

                print(f"Move {move}, h = {h_value}")
                print_state(neighbor)

                # Nếu gặp goal thì trả về ngay
                if neighbor == goal:
                    print("ĐÃ TÌM THẤY GOAL!")
                    return neighbor

                neighbor_states.append(neighbor)

        # Loại bỏ state trùng nhau
        unique_neighbors = []
        seen = set()

        for state in neighbor_states:
            if state not in seen:
                unique_neighbors.append(state)
                seen.add(state)

        # Sắp xếp theo h(n) tăng dần vì Manhattan càng nhỏ càng tốt
        unique_neighbors.sort(key=manhattan)

        # Lấy k trạng thái tốt nhất
        current_state_set = unique_neighbors[:k]

        print(f"Chọn {k} trạng thái tốt nhất:")

        for index, state in enumerate(current_state_set):
            print(f"Chosen State {index + 1}: h = {manhattan(state)}")
            print_state(state)

        # Nếu trong k state có goal thì dừng
        for state in current_state_set:
            if state == goal:
                print("ĐÃ TÌM THẤY GOAL!")
                return state

    print("KHÔNG TÌM THẤY GOAL TRONG SỐ BƯỚC GIỚI HẠN.")
    return None


# =========================
# 9. Chạy chương trình
# =========================

if __name__ == "__main__":
    result = local_beam_search(
        start=START,
        goal=GOAL,
        k=2,
        max_steps=20
    )

    print("===== KẾT QUẢ =====")
    if result is not None:
        print("Tìm thấy goal:")
        print_state(result)
    else:
        print("Không tìm thấy goal.")